# AI Distillation & PEFT LoRA Training Pipeline
This notebook demonstrates how to distill high-level reasoning from a massive teacher model (Llama-3.3-70B) down to a lightning-fast local student model (Qwen2.5-1.5B) using hybrid RAG generation and PEFT LoRA fine-tuning.

We use retro RPG game dialogue as a fun example use-case, but this exact distillation pipeline applies to any domain!

## 1. Setup & Installations
Install the necessary libraries for 4-bit quantization, PEFT LoRA, and API generation.

In [ ]:
!pip install -q transformers peft trl accelerate bitsandbytes datasets groq requests

## 2. RAG (Retrieval-Augmented Generation)
We dynamically pull contextual metadata (e.g., elemental typings from a live API) to enrich the teacher model's prompt.

In [ ]:
import requests
import json
import re

def get_entity_context(entity_name):
    try:
        res = requests.get(f"https://pokeapi.co/api/v2/pokemon/{entity_name.lower()}")
        if res.status_code == 200:
            types = [t["type"]["name"] for t in res.json()["types"]]
            return " and ".join(types)
    except:
        pass
    return None

def inject_rag_context(text):
    entity_mentions = re.findall(r'\b([A-Z]{3,})\b', text)
    context = []
    for entity in entity_mentions:
        etype = get_entity_context(entity)
        if etype:
            context.append(f"{entity} is a {etype} type.")
    return " ".join(context)

## 3. Hybrid Dataset Generation (Distillation)
We use the Groq API (Llama-3 70B) to generate the baseline dataset. If rate limits are hit, we seamlessly fallback to Qwen2.5-1.5B loaded locally on the Kaggle T4 GPUs to continue generating.

In [ ]:
import os
from groq import Groq
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup Groq Client (Add your GROQ_API_KEY to Kaggle Secrets)
# from kaggle_secrets import UserSecretsClient
# client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
client = Groq(api_key="your_api_key_here")

# Local Fallback Model setup
fallback_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = None
fallback_model = None

with open("text_database.json", "r") as f:
    game_texts = json.load(f)

dataset = []

for item in list(game_texts.items())[:10]: # Demo on first 10
    address, original_text = item
    rag_context = inject_rag_context(original_text)
    
    prompt = f"Rewrite this text to be funny and sarcastic. Original: {original_text}\nContext: {rag_context}"
    
    try:
        # 1. Try Groq API (Teacher)
        chat = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile"
        )
        response = chat.choices[0].message.content
    except Exception as e:
        print("Groq rate limit hit. Falling back to local Qwen2.5-1.5B on T4 GPUs...")
        # 2. Local Fallback
        if fallback_model is None:
            tokenizer = AutoTokenizer.from_pretrained(fallback_model_id)
            fallback_model = AutoModelForCausalLM.from_pretrained(
                fallback_model_id, 
                device_map="auto", 
                torch_dtype=torch.float16
            )
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = fallback_model.generate(**inputs, max_new_tokens=50)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, "").strip()
        
    dataset.append({"input": original_text, "output": response})

# Save distilled dataset
with open("distilled_dataset.json", "w") as f:
    json.dump(dataset, f)
print("Dataset Distillation Complete!")

## 4. Qwen2.5-1.5B LoRA Fine-Tuning (Student)
We take the high-quality dataset generated by Llama-3 and use it to train a fast, lightweight LoRA adapter for Qwen2.5-1.5B. We drastically reduce the VRAM footprint via 4-bit quantization.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

# Load Dataset
data = load_dataset("json", data_files="distilled_dataset.json", split="train")

def format_prompt(example):
    return f"<|im_start|>user\nRewrite this text: {example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"

# Load Model for Training (4-bit quantized to fit in T4 GPU)
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

# Setup LoRA
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Train
trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    formatting_func=format_prompt,
    args=TrainingArguments(
        output_dir="distilled_adapter",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        max_steps=100, # Example
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10
    )
)
trainer.train()

# Save the adapter
trainer.model.save_pretrained("distilled_adapter")
print("LoRA Adapter trained and saved successfully (approx 20MB)!")